# Editorial QA Agent – PPTX Review Pipeline

An agentic editorial QA pipeline for high-stakes, client-facing PowerPoint decks using **Microsoft Foundry Agents v2 SDK** with a **Gradio Web UI** and **production Azure deployment** path.

### What this does
1. **Extracts** visible slide text directly from PPTX (no OCR, no PDF conversion)
2. **Converts** to markdown with `# Slide N` markers preserved
3. **Analyzes visual consistency** deterministically — fonts, colors, positions, sizes, bold/italic, overlap, clutter (20+ checks)
4. **Reviews** content via adaptive sliding windows (single-pass for small decks, windowed for larger) using GPT-4o / GPT-5.x / Model Router
5. **4-pass editorial QA** per chunk: Terminology Inventory → Per-Slide QA → Cross-Slide Consistency → Verification (mandatory false-positive elimination)
6. **Merges & deduplicates** findings into a severity-ranked, slide-cited report

### Architecture
- **Foundry Agent with 6 FunctionTool calls** — the agent orchestrates extraction, visual analysis, chunking, review, storage, and merge
- **Hybrid approach** — deterministic Python analysis (visual) + LLM intelligence (editorial) — best of both worlds
- **Adaptive routing** — single-pass for small decks (<90K tokens), windowed with overlap for larger
- **7 review categories** — Spelling, Grammar, Punctuation, Terminology, Tone, Visual Consistency, Layout Quality
- **Zero false-positive policy** — Pass 4 verification drops unsupported findings; precision over recall
- **No OCR, no speaker notes/metadata** — only canvas-visible text via `python-pptx`

### Interfaces
- **Gradio Web UI** (`app.py`) — model selector, instructions editor, folder/upload/path input, summary dashboard, per-slide findings, full report
- **Jupyter Notebook** (`editorial_qa_agent.ipynb`) — step-by-step development and debugging pipeline

### Deployment
- **Local**: `python app.py` → http://127.0.0.1:7860
- **Production Azure**: Docker → Azure Container Apps (auto-scale 0→N) + Front Door + Key Vault + Managed Identity
- **IaC**: Bicep templates (`infra/main.bicep`) + one-command deploy (`deploy.ps1`)

### Key Files
| File | Purpose |
|---|---|
| `app.py` | Gradio web UI + full pipeline |
| `editorial_qa_agent.ipynb` | Development notebook |
| `generate_presentation.py` | Customer presentation generator (18 slides) |
| `Dockerfile` | Container image for Azure deployment |
| `infra/main.bicep` | Azure infrastructure as code |
| `deploy.ps1` | One-command Azure deployment script |
| `requirements.txt` | Python dependencies |

---

## Cell 1 — Environment Setup & Foundry Client Initialization

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1: Environment Setup & Foundry Client Initialization
# ═══════════════════════════════════════════════════════════════════════════════

import os
import json
import re
import logging
import pathlib
from dataclasses import dataclass, field, asdict
from typing import List, Tuple, Optional, Dict, Any
from dotenv import load_dotenv

# Azure AI Foundry SDK v2
from azure.identity import DefaultAzureCredential, AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool
from openai.types.responses.response_input_param import FunctionCallOutput

# PPTX extraction
from pptx import Presentation

# Display
from IPython.display import display, Markdown

# ── Resolve project root and load .env ──
PROJECT_ROOT = pathlib.Path(r'c:\Users\kapildhanger\OneDrive - Microsoft\Microsoft_Kapil\Azure_learning\Workshops\pptEditorialReview')
os.chdir(PROJECT_ROOT)
load_dotenv(str(PROJECT_ROOT / '.env'), override=True)

ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

# ── Configure logging ──
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("editorial_qa")

# ── Initialize Foundry client ──
# Use AzureCliCredential directly to avoid DefaultAzureCredential picking
# AzureDeveloperCliCredential from a different tenant
credential = AzureCliCredential()
project_client = AIProjectClient(endpoint=ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()

logger.info(f"Connected to Foundry project")
logger.info(f"Model deployment: {MODEL_DEPLOYMENT}")
print(f"✅ Foundry client initialized successfully")
print(f"   Model: {MODEL_DEPLOYMENT}")

2026-03-21 20:10:46,100 [INFO] Connected to Foundry project
2026-03-21 20:10:46,102 [INFO] Model deployment: gpt-4o


✅ Foundry client initialized successfully
   Model: gpt-4o


## Cell 2 — PPTX-to-Markdown Extraction + Visual Metadata

Two extraction pipelines defined here:

1. **Text extraction** (`pptx_to_markdown_slides`) — extracts only canvas-visible text and converts to markdown with `# Slide N` markers. Speaker notes, metadata, revision history, and timestamps are intentionally excluded.
2. **Visual metadata extraction** (`extract_slide_visual_metadata`) — captures per-shape layout data: position (inches), size, rotation, font properties (name, size, bold, italic, color), and fill colors for every shape on every slide.

The `# Slide N` markers are **critical infrastructure** — they enable deterministic citations in every downstream step. Never strip them.

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2: PPTX-to-Markdown Extraction + Visual Metadata
# ═══════════════════════════════════════════════════════════════════════════════

@dataclass
class SlideBlock:
    """Represents one slide's extracted markdown content."""
    slide_num: int
    markdown: str


def _clean(s: str) -> str:
    """Normalize whitespace: collapse non-breaking spaces, multiple spaces, excessive newlines."""
    s = s.replace("\xa0", " ")          # non-breaking space → regular space
    s = re.sub(r"[ ]+", " ", s)          # collapse multiple spaces
    s = re.sub(r"\n{3,}", "\n\n", s)    # collapse 3+ newlines to 2
    return s.strip()


def pptx_to_markdown_slides(pptx_path: str) -> List[SlideBlock]:
    """
    Extract visible slide text from a PPTX file and convert to markdown.
    
    Only extracts canvas-visible text (shapes with text_frame).
    Intentionally excludes: speaker notes, metadata, revision history, timestamps.
    
    Each slide gets a '# Slide N' header for deterministic citation.
    """
    prs = Presentation(pptx_path)
    blocks: List[SlideBlock] = []

    for i, slide in enumerate(prs.slides, start=1):
        lines = [f"# Slide {i}"]

        for shape in slide.shapes:
            # Only process shapes with text frames (skip images, charts, etc.)
            if not hasattr(shape, "text_frame"):
                continue
            tf = shape.text_frame
            if not tf or not tf.text:
                continue

            for paragraph in tf.paragraphs:
                txt = _clean(paragraph.text)
                if not txt:
                    continue
                # Indent level > 0 → bullet point
                if paragraph.level and paragraph.level > 0:
                    indent = "  " * (paragraph.level - 1)
                    lines.append(f"{indent}- {txt}")
                else:
                    lines.append(txt)

        md = _clean("\n".join(lines))
        # Only include slides that have content beyond the header
        if md and md != f"# Slide {i}":
            blocks.append(SlideBlock(i, md))

    return blocks


def join_slides(blocks: List[SlideBlock]) -> str:
    """Join slide blocks into a single markdown document with double-newline separators."""
    return "\n\n\n".join(b.markdown for b in blocks)


# ═══════════════════════════════════════════════════════════════════════════════
# Visual / layout metadata extraction
# ═══════════════════════════════════════════════════════════════════════════════

def _emu_to_inches(emu) -> float:
    """Convert EMU (English Metric Units) to inches. 914400 EMU = 1 inch."""
    if emu is None:
        return 0.0
    return round(emu / 914400, 2)


def _color_hex(color_obj) -> str | None:
    """Safely extract hex color string from a pptx color object."""
    try:
        if color_obj and color_obj.type is not None:
            return str(color_obj.rgb) if hasattr(color_obj, 'rgb') and color_obj.rgb else None
    except Exception:
        pass
    return None


def _font_props(font) -> dict:
    """Extract font properties from a pptx font object."""
    props = {}
    try:
        if font.name:
            props["name"] = font.name
        if font.size:
            props["size_pt"] = round(font.size.pt, 1)
        if font.bold is not None:
            props["bold"] = font.bold
        if font.italic is not None:
            props["italic"] = font.italic
        c = _color_hex(font.color)
        if c:
            props["color"] = c
    except Exception:
        pass
    return props


def extract_slide_visual_metadata(pptx_path: str) -> List[dict]:
    """
    Extract visual/layout metadata for every slide in a PPTX file.

    For each shape on each slide, captures:
      - position (left, top in inches), size (width, height in inches)
      - shape type, rotation
      - font properties per text run (name, size, bold, italic, color)
      - fill color (solid fills only)
    """
    prs = Presentation(pptx_path)
    slides_meta = []

    for slide_idx, slide in enumerate(prs.slides, start=1):
        shapes_meta = []
        for shape in slide.shapes:
            shape_info = {
                "shape_type": str(shape.shape_type) if shape.shape_type else "unknown",
                "name": shape.name or "",
                "left_in": _emu_to_inches(shape.left),
                "top_in": _emu_to_inches(shape.top),
                "width_in": _emu_to_inches(shape.width),
                "height_in": _emu_to_inches(shape.height),
                "rotation": shape.rotation if shape.rotation else 0,
            }

            # Fill color
            try:
                fill = shape.fill
                if fill and fill.type is not None:
                    fc = _color_hex(fill.fore_color)
                    if fc:
                        shape_info["fill_color"] = fc
            except Exception:
                pass

            # Text / font properties
            if hasattr(shape, "text_frame") and shape.text_frame:
                paras = []
                for para in shape.text_frame.paragraphs:
                    txt = _clean(para.text)
                    if not txt:
                        continue
                    fonts_in_para = []
                    for run in para.runs:
                        fp = _font_props(run.font)
                        if fp and fp not in fonts_in_para:
                            fonts_in_para.append(fp)
                    para_info = {"text": txt[:120]}
                    if fonts_in_para:
                        para_info["fonts"] = fonts_in_para
                    paras.append(para_info)
                if paras:
                    shape_info["paragraphs"] = paras

            shapes_meta.append(shape_info)

        slides_meta.append({
            "slide_num": slide_idx,
            "shape_count": len(shapes_meta),
            "shapes": shapes_meta,
        })

    return slides_meta


print("✅ PPTX extraction + visual metadata functions defined")

✅ PPTX extraction + visual metadata functions defined


## Cell 3 — Extraction Validation

Load a test deck and preview the extraction output to verify correctness.

In [13]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3: Extraction Validation
# ═══════════════════════════════════════════════════════════════════════════════

# ── Set your PPTX path here ──
PPTX_PATH = "test_decks/editorial_test.pptx"  # ← Update with your deck path

if os.path.exists(PPTX_PATH):
    slides = pptx_to_markdown_slides(PPTX_PATH)
    deck_md = join_slides(slides)
    
    print(f"📊 Extracted {len(slides)} slides with content")
    print(f"📝 Total characters: {len(deck_md):,}")
    print(f"📝 Total words: {len(deck_md.split()):,}")
    print(f"📝 Estimated tokens: {int(len(deck_md.split()) * 1.3):,}")
    print("\n" + "═" * 60)
    print("PREVIEW — First 3 slides:")
    print("═" * 60)
    for block in slides[:3]:
        print(f"\n{block.markdown}")
        print("─" * 40)
else:
    print(f"⚠️  No deck found at '{PPTX_PATH}' — update PPTX_PATH or run the test deck generator below.")

📊 Extracted 25 slides with content
📝 Total characters: 4,973
📝 Total words: 725
📝 Estimated tokens: 942

════════════════════════════════════════════════════════════
PREVIEW — First 3 slides:
════════════════════════════════════════════════════════════

# Slide 1
Digital Transformation Stratgy
Prepared for Contoso Ltd. | March 2026
────────────────────────────────────────

# Slide 2
Executive Summery
Contoso is at an inflection point in its digital journey.
This report outlines a phased aproach to modernization.
Key areas: cloud migration, data analytics, and customer experience.
────────────────────────────────────────

# Slide 3
Agenda
1. Current State Assessment
2. Strategic Recomendations
3. Implementation Roadmap
4. Investment Overview
────────────────────────────────────────


## Cell 4 — Adaptive Sliding Window Chunking

**Adaptive routing:**
- If total tokens < ~90k → **single-pass** (entire deck as one chunk) — better cross-slide consistency
- Otherwise → **windowed** with overlap to preserve cross-slide context

**Defaults:** `window_size=1200` words, `overlap=250` words

**Tuning:**
- Reduce overlap to 150–200 if results are repetitive
- Increase overlap to 300 or window to 1500 if cross-slide drift is missed

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4: Adaptive Sliding Window Chunking
# ═══════════════════════════════════════════════════════════════════════════════

# ── Configuration ──
SINGLE_PASS_TOKEN_THRESHOLD = 90_000   # Below this → single-pass mode
DEFAULT_WINDOW_SIZE = 1200             # Words per window
DEFAULT_OVERLAP = 250                  # Overlap words between windows


def estimate_token_count(text: str) -> int:
    """Rough token estimate: words × 1.3 (conservative for English text)."""
    return int(len(text.split()) * 1.3)


def sliding_window_chunks(
    text: str,
    window_size: int = DEFAULT_WINDOW_SIZE,
    overlap: int = DEFAULT_OVERLAP,
) -> List[Tuple[int, int, str]]:
    """
    Split text into overlapping word-based windows.
    
    Returns list of (start_word_idx, end_word_idx, chunk_text) tuples.
    The '# Slide N' markers survive into every chunk for citation support.
    """
    words = text.split()
    chunks: List[Tuple[int, int, str]] = []
    start = 0

    while start < len(words):
        end = min(start + window_size, len(words))
        chunk = " ".join(words[start:end])
        chunks.append((start, end, chunk))
        if end == len(words):
            break
        start = max(0, end - overlap)

    return chunks


def validate_chunks(chunks: List[Tuple[int, int, str]]) -> bool:
    """Verify every chunk contains at least one '# Slide N' marker."""
    slide_pattern = re.compile(r"# Slide \d+")
    all_valid = True
    for i, (start, end, chunk) in enumerate(chunks):
        if not slide_pattern.search(chunk):
            logger.warning(f"Chunk {i} (words {start}-{end}) has NO slide marker!")
            all_valid = False
    return all_valid


def adaptive_chunk(deck_md: str, window_size: int = DEFAULT_WINDOW_SIZE, overlap: int = DEFAULT_OVERLAP) -> List[Tuple[int, int, str]]:
    """
    Adaptive routing: single-pass for small decks, windowed for larger.
    
    Returns a list of chunks — either a single chunk (full deck) or multiple windows.
    """
    tokens = estimate_token_count(deck_md)
    words = deck_md.split()
    
    if tokens < SINGLE_PASS_TOKEN_THRESHOLD:
        logger.info(f"Single-pass mode: {tokens:,} estimated tokens (threshold: {SINGLE_PASS_TOKEN_THRESHOLD:,})")
        return [(0, len(words), deck_md)]
    else:
        logger.info(f"Windowed mode: {tokens:,} estimated tokens exceeds threshold")
        chunks = sliding_window_chunks(deck_md, window_size, overlap)
        valid = validate_chunks(chunks)
        if valid:
            logger.info(f"Generated {len(chunks)} chunks, all contain slide markers ✓")
        else:
            logger.warning("Some chunks are missing slide markers — review window parameters")
        return chunks


print("✅ Adaptive chunking functions defined")

✅ Adaptive chunking functions defined


## Cells 5–6 — Function Tool Definitions & Registration

Six functions exposed to the Foundry agent as **FunctionTool** calls.
The system prompt prescribes a strict invocation order (extract → visual analysis → chunk → review → store → merge).

| Tool | Purpose |
|---|---|
| `extract_deck` | PPTX → markdown conversion with slide markers and metadata |
| `extract_deck_visual` | Raw shape/font/color/position metadata for visual review |
| `analyze_visual_consistency` | **Deterministic** visual & layout consistency analysis — computes dominant patterns and returns pre-formatted deviations ready for storage |
| `get_review_windows` | Adaptive chunking of the markdown |
| `store_chunk_findings` | Accumulate QA findings from each chunk (use `chunk_id=-1` for visual findings) |
| `merge_and_dedupe_findings` | Final deduplication and severity ranking |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELLS 5–6: Function Tool Definitions & Registration
# ═══════════════════════════════════════════════════════════════════════════════

# ── Shared state (notebook-scoped accumulator) ──
findings_accumulator: List[Dict[str, Any]] = []
extracted_deck_cache: Dict[str, str] = {}  # pptx_path → markdown


# ────────────────────────────────────────────────────────────────────────────
# Tool 1: extract_deck
# ────────────────────────────────────────────────────────────────────────────
def extract_deck(pptx_path: str) -> str:
    """Extract visible text from a PPTX file, convert to markdown with slide markers."""
    blocks = pptx_to_markdown_slides(pptx_path)
    deck_md = join_slides(blocks)
    extracted_deck_cache[pptx_path] = deck_md
    
    return json.dumps({
        "slide_count": len(blocks),
        "word_count": len(deck_md.split()),
        "estimated_tokens": estimate_token_count(deck_md),
        "markdown": deck_md
    })


# ────────────────────────────────────────────────────────────────────────────
# Tool 2: extract_deck_visual
# ────────────────────────────────────────────────────────────────────────────
def extract_deck_visual(pptx_path: str) -> str:
    """Extract visual/layout metadata (fonts, colors, positions, sizes) for all slides."""
    try:
        meta = extract_slide_visual_metadata(pptx_path)
        return json.dumps({"slide_count": len(meta), "slides": meta})
    except Exception as e:
        logger.error(f"Visual extraction failed: {e}")
        return json.dumps({"error": str(e)})


# ────────────────────────────────────────────────────────────────────────────
# Tool 2b: analyze_visual_consistency (deterministic pre-processing)
# ────────────────────────────────────────────────────────────────────────────
def analyze_visual_consistency(pptx_path: str) -> str:
    """
    Deterministic visual consistency analysis.

    Extracts visual metadata, computes the dominant pattern for titles and body,
    then returns ONLY the deviations — pre-computed so the LLM doesn't have to
    scan thousands of tokens of raw shape data.
    """
    from collections import Counter
    meta = extract_slide_visual_metadata(pptx_path)

    # ── Classify shapes per slide: title = first text shape, body = subsequent ──
    slides_info = []
    for s in meta:
        text_shapes = [sh for sh in s["shapes"] if "paragraphs" in sh]
        deco_shapes = [sh for sh in s["shapes"] if "paragraphs" not in sh and sh.get("fill_color")]
        title_sh = text_shapes[0] if text_shapes else None
        body_shs = text_shapes[1:] if len(text_shapes) > 1 else []
        slides_info.append({
            "slide_num": s["slide_num"],
            "shape_count": s["shape_count"],
            "title": title_sh,
            "body": body_shs,
            "deco": deco_shapes,
            "all_shapes": s["shapes"],
        })

    def _first_font(shape):
        if not shape or "paragraphs" not in shape:
            return {}
        for p in shape["paragraphs"]:
            if "fonts" in p and p["fonts"]:
                return p["fonts"][0]
        return {}

    # ── Compute dominant title pattern ──
    title_fonts = Counter()
    title_sizes = Counter()
    title_bolds = Counter()
    title_italics = Counter()
    title_colors = Counter()
    title_lefts = Counter()
    title_tops = Counter()
    title_widths = Counter()

    for si in slides_info:
        f = _first_font(si["title"])
        if not f:
            continue
        if f.get("name"):   title_fonts[f["name"]] += 1
        if f.get("size_pt"): title_sizes[f["size_pt"]] += 1
        if "bold" in f:      title_bolds[f["bold"]] += 1
        if "italic" in f:    title_italics[f["italic"]] += 1
        if f.get("color"):   title_colors[f["color"]] += 1
        if si["title"]:
            title_lefts[si["title"].get("left_in")] += 1
            title_tops[si["title"].get("top_in")] += 1
            title_widths[si["title"].get("width_in")] += 1

    dom_t = {
        "font": title_fonts.most_common(1)[0][0] if title_fonts else None,
        "size": title_sizes.most_common(1)[0][0] if title_sizes else None,
        "bold": title_bolds.most_common(1)[0][0] if title_bolds else None,
        "italic": title_italics.most_common(1)[0][0] if title_italics else None,
        "color": title_colors.most_common(1)[0][0] if title_colors else None,
        "left": title_lefts.most_common(1)[0][0] if title_lefts else None,
        "top": title_tops.most_common(1)[0][0] if title_tops else None,
        "width": title_widths.most_common(1)[0][0] if title_widths else None,
    }

    # ── Compute dominant body pattern ──
    body_fonts = Counter()
    body_sizes = Counter()
    body_bolds = Counter()
    body_italics = Counter()
    body_colors = Counter()

    for si in slides_info:
        for bsh in si["body"]:
            f = _first_font(bsh)
            if not f:
                continue
            if f.get("name"):   body_fonts[f["name"]] += 1
            if f.get("size_pt"): body_sizes[f["size_pt"]] += 1
            if "bold" in f:      body_bolds[f["bold"]] += 1
            if "italic" in f:    body_italics[f["italic"]] += 1
            if f.get("color"):   body_colors[f["color"]] += 1

    dom_b = {
        "font": body_fonts.most_common(1)[0][0] if body_fonts else None,
        "size": body_sizes.most_common(1)[0][0] if body_sizes else None,
        "bold": body_bolds.most_common(1)[0][0] if body_bolds else None,
        "italic": body_italics.most_common(1)[0][0] if body_italics else None,
        "color": body_colors.most_common(1)[0][0] if body_colors else None,
    }

    # ── Dominant accent fill ──
    fill_colors = Counter()
    for si in slides_info:
        for d in si["deco"]:
            if d.get("fill_color"):
                fill_colors[d["fill_color"]] += 1
    dom_fill = fill_colors.most_common(1)[0][0] if fill_colors else None

    # ── Average shape count (for clutter detection) ──
    counts = [si["shape_count"] for si in slides_info]
    avg_shapes = sum(counts) / len(counts) if counts else 0

    # ══════════════════════════════════════════════════════════════════════
    # Find deviations
    # ══════════════════════════════════════════════════════════════════════
    issues = []

    for si in slides_info:
        sn = si["slide_num"]
        tf = _first_font(si["title"])
        tsh = si["title"]

        # ── Cross-slide title font deviations ──
        if tf.get("name") and dom_t["font"] and tf["name"] != dom_t["font"]:
            issues.append({"slide": sn, "type": "Visual", "element": "title",
                           "check": "font_family",
                           "found": tf["name"], "expected": dom_t["font"],
                           "detail": f"Title font '{tf['name']}' vs dominant '{dom_t['font']}'"})
        if tf.get("size_pt") and dom_t["size"] and tf["size_pt"] != dom_t["size"]:
            issues.append({"slide": sn, "type": "Visual", "element": "title",
                           "check": "font_size",
                           "found": tf["size_pt"], "expected": dom_t["size"],
                           "detail": f"Title size {tf['size_pt']}pt vs dominant {dom_t['size']}pt"})
        if "bold" in tf and dom_t["bold"] is not None and tf["bold"] != dom_t["bold"]:
            issues.append({"slide": sn, "type": "Visual", "element": "title",
                           "check": "bold",
                           "found": tf.get("bold"), "expected": dom_t["bold"],
                           "detail": f"Title bold={tf['bold']} vs dominant bold={dom_t['bold']}"})
        if "italic" in tf and dom_t["italic"] is not None and tf["italic"] != dom_t["italic"]:
            issues.append({"slide": sn, "type": "Visual", "element": "title",
                           "check": "italic",
                           "found": tf.get("italic"), "expected": dom_t["italic"],
                           "detail": f"Title italic={tf['italic']} vs dominant italic={dom_t['italic']}"})
        if tf.get("color") and dom_t["color"] and tf["color"] != dom_t["color"]:
            issues.append({"slide": sn, "type": "Visual", "element": "title",
                           "check": "text_color",
                           "found": tf["color"], "expected": dom_t["color"],
                           "detail": f"Title color #{tf['color']} vs dominant #{dom_t['color']}"})

        # ── Cross-slide title position/size deviations ──
        if tsh and dom_t["top"] is not None:
            if abs((tsh.get("top_in") or 0) - dom_t["top"]) > 0.3:
                issues.append({"slide": sn, "type": "Layout", "element": "title",
                               "check": "position_top",
                               "found": tsh.get("top_in"), "expected": dom_t["top"],
                               "detail": f"Title top={tsh.get('top_in')}\" vs dominant {dom_t['top']}\""})
        if tsh and dom_t["left"] is not None:
            if abs((tsh.get("left_in") or 0) - dom_t["left"]) > 0.3:
                issues.append({"slide": sn, "type": "Layout", "element": "title",
                               "check": "position_left",
                               "found": tsh.get("left_in"), "expected": dom_t["left"],
                               "detail": f"Title left={tsh.get('left_in')}\" vs dominant {dom_t['left']}\""})
        if tsh and dom_t["width"] is not None:
            if abs((tsh.get("width_in") or 0) - dom_t["width"]) > 1.0:
                issues.append({"slide": sn, "type": "Visual", "element": "title",
                               "check": "shape_width",
                               "found": tsh.get("width_in"), "expected": dom_t["width"],
                               "detail": f"Title width={tsh.get('width_in')}\" vs dominant {dom_t['width']}\""})

        # ── Cross-slide body font deviations (consolidated per slide) ──
        body_dev_fonts = set()
        body_dev_sizes = set()
        body_dev_bolds = set()
        body_dev_italics = set()
        body_dev_colors = set()
        for bsh in si["body"]:
            bf = _first_font(bsh)
            if bf.get("name") and dom_b["font"] and bf["name"] != dom_b["font"]:
                body_dev_fonts.add(bf["name"])
            if bf.get("size_pt") and dom_b["size"] and bf["size_pt"] != dom_b["size"]:
                body_dev_sizes.add(bf["size_pt"])
            if "bold" in bf and dom_b["bold"] is not None and bf["bold"] != dom_b["bold"]:
                body_dev_bolds.add(bf["bold"])
            if "italic" in bf and dom_b["italic"] is not None and bf["italic"] != dom_b["italic"]:
                body_dev_italics.add(bf["italic"])
            if bf.get("color") and dom_b["color"] and bf["color"] != dom_b["color"]:
                body_dev_colors.add(bf["color"])
        if body_dev_fonts:
            issues.append({"slide": sn, "type": "Visual", "element": "body",
                           "check": "font_family",
                           "found": sorted(body_dev_fonts), "expected": dom_b["font"],
                           "detail": f"Body font(s) {sorted(body_dev_fonts)} vs dominant '{dom_b['font']}'"})
        if body_dev_sizes:
            issues.append({"slide": sn, "type": "Visual", "element": "body",
                           "check": "font_size",
                           "found": sorted(body_dev_sizes), "expected": dom_b["size"],
                           "detail": f"Body size(s) {sorted(body_dev_sizes)}pt vs dominant {dom_b['size']}pt"})
        if body_dev_bolds:
            issues.append({"slide": sn, "type": "Visual", "element": "body",
                           "check": "bold",
                           "found": sorted(body_dev_bolds), "expected": dom_b["bold"],
                           "detail": f"Body bold deviation vs dominant bold={dom_b['bold']}"})
        if body_dev_italics:
            issues.append({"slide": sn, "type": "Visual", "element": "body",
                           "check": "italic",
                           "found": sorted(body_dev_italics), "expected": dom_b["italic"],
                           "detail": f"Body italic deviation vs dominant italic={dom_b['italic']}"})
        if body_dev_colors:
            issues.append({"slide": sn, "type": "Visual", "element": "body",
                           "check": "text_color",
                           "found": sorted(body_dev_colors), "expected": dom_b["color"],
                           "detail": f"Body color(s) {['#'+c for c in sorted(body_dev_colors)]} vs dominant #{dom_b['color']}"})

        # ── Within-slide font/size/color mixing among body shapes ──
        if len(si["body"]) >= 2:
            fonts_on_slide = set()
            sizes_on_slide = set()
            colors_on_slide = set()
            for bsh in si["body"]:
                bf = _first_font(bsh)
                if bf.get("name"):   fonts_on_slide.add(bf["name"])
                if bf.get("size_pt"): sizes_on_slide.add(bf["size_pt"])
                if bf.get("color"):  colors_on_slide.add(bf["color"])
            if len(fonts_on_slide) > 1:
                issues.append({"slide": sn, "type": "Visual", "element": "body",
                               "check": "within_slide_font_mix",
                               "found": sorted(fonts_on_slide), "expected": "same font",
                               "detail": f"Within-slide font mixing: {sorted(fonts_on_slide)}"})
            if len(sizes_on_slide) > 1:
                issues.append({"slide": sn, "type": "Visual", "element": "body",
                               "check": "within_slide_size_mix",
                               "found": sorted(sizes_on_slide), "expected": "same size",
                               "detail": f"Within-slide size mixing: {sorted(sizes_on_slide)}pt"})
            if len(colors_on_slide) > 1:
                issues.append({"slide": sn, "type": "Visual", "element": "body",
                               "check": "within_slide_color_mix",
                               "found": sorted(colors_on_slide), "expected": "same color",
                               "detail": f"Within-slide color mixing: {['#'+c for c in sorted(colors_on_slide)]}"})

        # ── Accent fill color deviation ──
        for d in si["deco"]:
            if d.get("fill_color") and dom_fill and d["fill_color"] != dom_fill:
                issues.append({"slide": sn, "type": "Visual", "element": "accent",
                               "check": "fill_color",
                               "found": d["fill_color"], "expected": dom_fill,
                               "detail": f"Accent fill #{d['fill_color']} vs dominant #{dom_fill}"})

        # ── Layout: Clutter ──
        if si["shape_count"] > avg_shapes * 2 and si["shape_count"] > 6:
            issues.append({"slide": sn, "type": "Layout", "element": "slide",
                           "check": "clutter",
                           "found": si["shape_count"], "expected": f"avg {avg_shapes:.0f}",
                           "detail": f"{si['shape_count']} shapes vs deck avg {avg_shapes:.0f}"})

        # ── Layout: Overlap detection ──
        all_shapes = ([si["title"]] if si["title"] else []) + si["body"]
        for i_a in range(len(all_shapes)):
            for i_b in range(i_a + 1, len(all_shapes)):
                a, b = all_shapes[i_a], all_shapes[i_b]
                a_r = a.get("left_in", 0) + a.get("width_in", 0)
                a_bot = a.get("top_in", 0) + a.get("height_in", 0)
                b_r = b.get("left_in", 0) + b.get("width_in", 0)
                b_bot = b.get("top_in", 0) + b.get("height_in", 0)
                if (a.get("left_in", 0) < b_r and b.get("left_in", 0) < a_r and
                    a.get("top_in", 0) < b_bot and b.get("top_in", 0) < a_bot):
                    issues.append({"slide": sn, "type": "Layout", "element": "shapes",
                                   "check": "overlap",
                                   "found": f"{a.get('name','')} ↔ {b.get('name','')}",
                                   "expected": "no overlap",
                                   "detail": f"Shapes '{a.get('name','')}' and '{b.get('name','')}' overlap"})

        # ── Layout: Crammed margins ──
        check_shapes = ([si["title"]] if si["title"] else []) + si["body"]
        for sh in check_shapes:
            if not sh:
                continue
            l = sh.get("left_in", 1)
            t = sh.get("top_in", 1)
            if l < 0.3 or t < 0.3:
                issues.append({"slide": sn, "type": "Layout", "element": sh.get("name", "shape"),
                               "check": "crammed_margin",
                               "found": f"left={l}\", top={t}\"", "expected": ">0.3\" margin",
                               "detail": f"Shape '{sh.get('name','')}' at ({l}\", {t}\") — crammed to edge"})

        # ── Layout: Misaligned columns ──
        if len(si["body"]) >= 3:
            tops = [bsh.get("top_in", 0) for bsh in si["body"]]
            if max(tops) - min(tops) > 0.3:
                issues.append({"slide": sn, "type": "Layout", "element": "columns",
                               "check": "misaligned_columns",
                               "found": tops, "expected": "same top values",
                               "detail": f"Column tops vary: {tops} (diff={max(tops)-min(tops):.1f}\")"})

        # ── Layout: Rotation ──
        for sh in si["all_shapes"]:
            if sh.get("rotation") and sh["rotation"] != 0:
                issues.append({"slide": sn, "type": "Layout", "element": sh.get("name", "shape"),
                               "check": "rotation",
                               "found": sh["rotation"], "expected": 0,
                               "detail": f"Shape '{sh.get('name','')}' rotated {sh['rotation']}°"})

    # ══════════════════════════════════════════════════════════════════════
    # Filter: skip slide 1 (cover) and last slide (closing)
    # ══════════════════════════════════════════════════════════════════════
    last_slide = max(si["slide_num"] for si in slides_info) if slides_info else 0
    filtered = [i for i in issues
                if i["slide"] != 1 and i["slide"] != last_slide]

    # ── Severity assignment ──
    def _severity(iss):
        check = iss["check"]
        element = iss["element"]
        if check == "overlap":
            return "Critical"
        if check in ("font_family", "font_size", "text_color") and element == "title":
            return "High"
        if check in ("bold", "italic") and element == "title":
            return "High"
        if check in ("font_family", "font_size", "text_color", "bold", "italic") and element == "body":
            return "Medium"
        if check.startswith("within_slide"):
            return "Medium"
        if check == "fill_color":
            return "Medium"
        if check in ("shape_width",):
            return "Medium"
        if check in ("clutter", "crammed_margin", "misaligned_columns"):
            return "Medium"
        if check in ("position_top", "position_left"):
            return "Medium"
        if check == "rotation":
            return "Low"
        return "Medium"

    # ── Build pre-formatted findings ready for store_chunk_findings ──
    formatted = []
    for iss in filtered:
        formatted.append({
            "slides": f"Slide {iss['slide']}",
            "evidence": iss["detail"],
            "issue": f"{iss['element'].title()} {iss['check'].replace('_', ' ')} — deviates from dominant pattern",
            "flag": _severity(iss),
            "remediation": f"Change to match dominant pattern ({iss['expected']})",
            "category": iss["type"],
        })

    return json.dumps({
        "dominant_title": dom_t,
        "dominant_body": dom_b,
        "dominant_accent_fill": dom_fill,
        "avg_shape_count": round(avg_shapes, 1),
        "total_inconsistencies": len(filtered),
        "findings_json": json.dumps(formatted),
    })


# ────────────────────────────────────────────────────────────────────────────
# Tool 3: get_review_windows
# ────────────────────────────────────────────────────────────────────────────
def get_review_windows(markdown: str, window_size: int = 1200, overlap: int = 250) -> str:
    """Split markdown into overlapping review windows. Uses single-pass for small decks."""
    chunks = adaptive_chunk(markdown, window_size, overlap)
    
    result = []
    for idx, (start, end, text) in enumerate(chunks):
        slide_nums = [int(m) for m in re.findall(r"# Slide (\d+)", text)]
        result.append({
            "chunk_id": idx,
            "start_word": start,
            "end_word": end,
            "word_count": end - start,
            "slides_covered": slide_nums,
            "text": text
        })
    
    return json.dumps({
        "total_chunks": len(result),
        "mode": "single_pass" if len(result) == 1 else "windowed",
        "chunks": result
    })


# ────────────────────────────────────────────────────────────────────────────
# Tool 4: store_chunk_findings
# ────────────────────────────────────────────────────────────────────────────
def store_chunk_findings(chunk_id: int, findings_json: str) -> str:
    """Store QA findings from one chunk into the accumulator."""
    try:
        findings = json.loads(findings_json)
        if not isinstance(findings, list):
            findings = [findings]
    except json.JSONDecodeError:
        return json.dumps({"error": "Invalid JSON in findings", "stored": False})
    
    for finding in findings:
        finding["source_chunk"] = chunk_id
    
    logger.info(f"store_chunk_findings(chunk_id={chunk_id}): {len(findings)} findings")
    findings_accumulator.extend(findings)
    
    return json.dumps({
        "stored": True,
        "chunk_id": chunk_id,
        "findings_in_chunk": len(findings),
        "total_findings_so_far": len(findings_accumulator)
    })


# ────────────────────────────────────────────────────────────────────────────
# Tool 5: merge_and_dedupe_findings
# ────────────────────────────────────────────────────────────────────────────
SEVERITY_RANK = {"Critical": 4, "High": 3, "Medium": 2, "Low": 1}


def merge_and_dedupe_findings() -> str:
    """Merge all accumulated findings, deduplicate, and keep highest severity."""
    if not findings_accumulator:
        return json.dumps({"merged_findings": [], "total": 0, "duplicates_removed": 0})
    
    seen: Dict[str, Dict[str, Any]] = {}
    
    for finding in findings_accumulator:
        slides_key = str(finding.get("slides", finding.get("slide", ""))).strip().lower()
        issue_key = str(finding.get("issue", finding.get("category", ""))).strip().lower()
        evidence_key = str(finding.get("evidence", finding.get("remediation", ""))).strip().lower()
        dedup_key = f"{slides_key}|{issue_key}|{evidence_key}"
        
        current_severity = str(finding.get("flag", finding.get("severity", "Low")))
        current_rank = SEVERITY_RANK.get(current_severity, 0)
        
        if dedup_key in seen:
            existing_severity = str(seen[dedup_key].get("flag", seen[dedup_key].get("severity", "Low")))
            existing_rank = SEVERITY_RANK.get(existing_severity, 0)
            if current_rank > existing_rank:
                seen[dedup_key] = finding
        else:
            seen[dedup_key] = finding
    
    merged = list(seen.values())
    
    def sort_key(f):
        sev = SEVERITY_RANK.get(str(f.get("flag", f.get("severity", "Low"))), 0)
        slide_str = str(f.get("slides", f.get("slide", "999")))
        slide_num = int(re.search(r"\d+", slide_str).group()) if re.search(r"\d+", slide_str) else 999
        return (-sev, slide_num)
    
    merged.sort(key=sort_key)
    duplicates_removed = len(findings_accumulator) - len(merged)
    
    return json.dumps({
        "merged_findings": merged,
        "total": len(merged),
        "duplicates_removed": duplicates_removed
    })


# ── Map tool names to their Python functions ──
TOOL_FUNCTIONS = {
    "extract_deck": extract_deck,
    "extract_deck_visual": extract_deck_visual,
    "analyze_visual_consistency": analyze_visual_consistency,
    "get_review_windows": get_review_windows,
    "store_chunk_findings": store_chunk_findings,
    "merge_and_dedupe_findings": merge_and_dedupe_findings,
}


# ════════════════════════════════════════════════════════════════════════════
# FunctionTool declarations for the Foundry Agent (JSON Schema)
# ════════════════════════════════════════════════════════════════════════════

tool_extract_deck = FunctionTool(
    name="extract_deck",
    description="Extract visible text from a PPTX file, convert to markdown with slide markers.",
    parameters={
        "type": "object",
        "properties": {
            "pptx_path": {"type": "string", "description": "Path to the PPTX file"}
        },
        "required": ["pptx_path"],
        "additionalProperties": False
    },
    strict=True,
)

tool_extract_deck_visual = FunctionTool(
    name="extract_deck_visual",
    description="Extract raw visual/layout metadata from a PPTX file. Returns per-shape details. Only use when you need raw data; prefer analyze_visual_consistency for the review.",
    parameters={
        "type": "object",
        "properties": {
            "pptx_path": {"type": "string", "description": "Path to the PPTX file"}
        },
        "required": ["pptx_path"],
        "additionalProperties": False
    },
    strict=True,
)

tool_analyze_visual = FunctionTool(
    name="analyze_visual_consistency",
    description="Run deterministic visual & layout consistency analysis on a PPTX file. Returns dominant patterns and pre-formatted findings_json ready to pass directly to store_chunk_findings(chunk_id=-1). Use this as step 2 of the editorial QA process.",
    parameters={
        "type": "object",
        "properties": {
            "pptx_path": {"type": "string", "description": "Path to the PPTX file"}
        },
        "required": ["pptx_path"],
        "additionalProperties": False
    },
    strict=True,
)

tool_get_review_windows = FunctionTool(
    name="get_review_windows",
    description="Split markdown into overlapping review windows.",
    parameters={
        "type": "object",
        "properties": {
            "markdown": {"type": "string", "description": "Full markdown text"},
            "window_size": {"type": "integer", "description": "Words per window"},
            "overlap": {"type": "integer", "description": "Overlap words"}
        },
        "required": ["markdown", "window_size", "overlap"],
        "additionalProperties": False
    },
    strict=True,
)

tool_store_chunk_findings = FunctionTool(
    name="store_chunk_findings",
    description="Store QA findings from one chunk. Use chunk_id=-1 for visual/layout findings.",
    parameters={
        "type": "object",
        "properties": {
            "chunk_id": {"type": "integer", "description": "Chunk ID (-1 for visual)"},
            "findings_json": {"type": "string", "description": "JSON array of findings"}
        },
        "required": ["chunk_id", "findings_json"],
        "additionalProperties": False
    },
    strict=True,
)

tool_merge_and_dedupe = FunctionTool(
    name="merge_and_dedupe_findings",
    description="Merge and deduplicate all findings.",
    parameters={
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    },
    strict=True,
)

ALL_TOOLS = [tool_extract_deck, tool_extract_deck_visual, tool_analyze_visual, tool_get_review_windows, tool_store_chunk_findings, tool_merge_and_dedupe]

print(f"✅ {len(ALL_TOOLS)} function tools defined and registered")
for t in ALL_TOOLS:
    print(f"   • {t.name}")

✅ 5 function tools defined and registered
   • extract_deck
   • extract_deck_visual
   • get_review_windows
   • store_chunk_findings
   • merge_and_dedupe_findings


## Cell 7 — Agent Creation with Editorial QA Prompt Contract

The system prompt is the **authoritative contract**. Every constraint prevents hallucination or drift.

**Do not simplify this prompt** — each rule exists for a reason:
- 7 must-flag categories (spelling, grammar, punctuation, terminology, tone, **visual consistency**, **layout quality**)
- 4-pass process (inventory → per-slide → cross-slide → **verification**)
- Mandatory `evidence` field — agent must quote exact erroneous text
- Zero tolerance for false positives — precision over recall
- Explicit exclusions: no flagging stats, bullet fragments, heading punctuation
- Severity flags: Critical / High / Medium / Low
- **Mandatory tool calls** — agent must call `store_chunk_findings` for every chunk (including visual with chunk_id=-1)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7: Agent Creation with Editorial QA Prompt Contract
# ═══════════════════════════════════════════════════════════════════════════════

SYSTEM_INSTRUCTIONS = """\
You are an editorial QA reviewer for high-stakes, client-facing deliverables (often PowerPoint).
Your job is to protect credibility by flagging errors and inconsistencies that a customer would notice.

You have access to function tools. Use them in this order:
1. Call extract_deck with the provided PPTX path to get the markdown.
2. Call get_review_windows with the markdown to split it into review chunks.
3. For EACH chunk, perform the editorial review described below, then call store_chunk_findings with the results.
4. After ALL chunks are reviewed, call merge_and_dedupe_findings to get the final output.

EDITORIAL REVIEW PROCESS (apply to each chunk):

Analyze only customer-visible content. Do not use speaker notes, comments, revision history, timestamps, or file metadata.

Must-flag categories (do not skip any):
1) Spelling: typos, misspellings, incorrect word forms, obviously wrong proper nouns in context.
2) Grammar: agreement, tense consistency, broken sentence structure, incorrect articles, awkward fragments.
3) Punctuation: missing punctuation, inconsistent style, broken quotes, inconsistent hyphenation (e.g. "real time" vs "real-time").
4) Terminology consistency: same concept named multiple ways across slides (capitalization, hyphenation, acronyms, product names, metrics).
5) Tone and voice consistency: shifts suggesting multiple authors or stitched content (formal vs casual, marketing hype vs neutral, inconsistent we/you/they, inconsistent certainty).

Constraints:
- ZERO TOLERANCE FOR FALSE POSITIVES. It is far worse to flag a non-issue than to miss a real one. When in doubt, do NOT flag it.
- Do not invent issues. Only flag what is unambiguously wrong in the extracted text.
- Every finding MUST include the exact verbatim text that is wrong in the "evidence" field (copy-paste from the markdown). If you cannot quote the exact erroneous text, do not report the finding.
- Do not flag layout, formatting, or visual design choices — you only see extracted text, not how it is rendered on the slide.
- Do not flag statistics, percentages, or numeric data as errors unless there is a clear internal contradiction (e.g., "ROI is 340%" on one slide and "ROI is 280%" on another).
- Do not flag sentence fragments or bullet-point style phrasing — PowerPoint slides are not prose documents. Bullet points, headlines, and short phrases are normal and expected. Specifically:
  * Do NOT flag bullet points or headings for missing periods, missing articles ("a", "the"), or missing conjunctions.
  * Do NOT flag short noun phrases like "Observability and agent controls" or "Enterprise-grade security" — these are standard slide labels, not broken sentences.
  * Do NOT suggest adding periods to the end of bullet points or headings.
  * Do NOT flag lack of full sentence structure in bullet items — they are meant to be concise.
- Do not flag punctuation in headings, labels, or statistics (e.g., "85%" is correct, not a punctuation error).
- Do not rewrite the document. Provide targeted remediation only.
- If a slide has no issues, do not list it.
- Only flag contradictions if both statements are visible and you can cite the slide numbers involved.

For each chunk, perform this 4-pass process:
Pass 1 (Inventory): Build a Canonical Glossary from recurring key terms, acronyms, product names, and metrics. Choose canonical form based on most common or most formal usage. Identify dominant tone in 1-2 sentences.
Pass 2 (Per-slide QA): Scan each slide for spelling, grammar, and punctuation issues. Flag tone anomalies only when clearly divergent from surrounding slides.
Pass 3 (Cross-slide consistency): Using your Canonical Glossary and dominant voice, flag terminology drift, inconsistent acronym usage, and tone drift.
Pass 4 (VERIFICATION — MANDATORY): Before storing findings, re-read each finding and ask: "Can I point to the EXACT wrong text and explain precisely why it is wrong?" Drop any finding where the answer is no. Drop any finding that is subjective, stylistic preference, or layout-related. This pass should eliminate at least 30% of initial findings. Precision matters more than recall.

When calling store_chunk_findings, provide a JSON array of objects:
[
  {
    "slides": "Slide 5" or "Slides 5, 12",
    "evidence": "exact verbatim text from the slide that contains the error (copy-paste)",
    "issue": "description of the issue — explain WHY the evidence text is wrong",
    "flag": "Critical" or "High" or "Medium" or "Low",
    "remediation": "suggested fix",
    "category": "Spelling" or "Grammar" or "Punctuation" or "Terminology" or "Tone"
  }
]

IMPORTANT: The "evidence" field is REQUIRED. If you cannot provide the exact erroneous text, do not include the finding.

Severity flags:
- Critical: would cause embarrassment or immediate credibility loss
- High: clear errors a customer will notice
- Medium: consistency issues that erode polish
- Low: minor style preferences (use sparingly)

Remediation rules:
- Typos: show corrected word or phrase
- Grammar/Punctuation: show corrected fragment
- Terminology: specify canonical term, suggest find/replace + spot check
- Tone drift: describe the drift in one sentence, suggest target voice (do not rewrite)

After all chunks are reviewed and findings stored, call merge_and_dedupe_findings and present the final results.
"""

# ── Create the agent ──
agent = project_client.agents.create_version(
    agent_name="editorial-qa-reviewer",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT,
        instructions=SYSTEM_INSTRUCTIONS,
        tools=ALL_TOOLS,
    ),
)

logger.info(f"Agent created: name={agent.name}, version={agent.version}, id={agent.id}")
print(f"✅ Agent created: {agent.name} (version: {agent.version})")

2026-03-21 20:11:04,468 [INFO] AzureCliCredential.get_token_info succeeded
2026-03-21 20:11:04,469 [INFO] Request URL: 'https://kd-foundry-resource.services.ai.azure.com/api/projects/kd-proj/agents/editorial-qa-reviewer/versions?api-version=REDACTED'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '11726'
    'Accept': 'application/json'
    'x-ms-client-request-id': '9cae2daa-2583-11f1-beeb-6ca1005710f1'
    'User-Agent': 'azsdk-python-ai-projects/2.0.1 Python/3.12.10 (Windows-11-10.0.26200-SP0)'
    'Authorization': 'REDACTED'
A body is sent with the request
2026-03-21 20:11:05,231 [INFO] Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; charset=utf-8'
    'content-encoding': 'REDACTED'
    'vary': 'REDACTED'
    'request-context': 'REDACTED'
    'x-ms-response-type': 'REDACTED'
    'x-request-id': 'REDACTED'
    'api-supported-versions': 'REDACTED'
    'azureml-served-

✅ Agent created: editorial-qa-reviewer (version: 3)


## Cells 8–9 — Agentic Orchestration Loop

The orchestration follows this flow:
1. Send the user message asking the agent to review a deck
2. The agent calls function tools → we execute them locally → return results
3. Loop until the agent produces its final response

This uses the **Foundry Agents v2 SDK** pattern:
- `openai_client.responses.create()` for each turn
- `FunctionCallOutput` to return tool results
- `previous_response_id` to maintain conversation continuity

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELLS 8–9: Agentic Orchestration Loop with Tool Call Handling
# ═══════════════════════════════════════════════════════════════════════════════

import time

MAX_TOOL_ROUNDS = 50      # Safety limit to prevent infinite loops
MAX_RETRIES = 3           # Retries per API call on transient errors
RETRY_BASE_DELAY = 2      # Seconds — exponential backoff base


def execute_tool_call(name: str, arguments: str) -> str:
    """
    Execute a function tool call locally and return the result as a JSON string.
    """
    func = TOOL_FUNCTIONS.get(name)
    if not func:
        return json.dumps({"error": f"Unknown tool: {name}"})
    
    try:
        args = json.loads(arguments)
        result = func(**args)
        return result
    except Exception as e:
        logger.error(f"Tool '{name}' failed: {e}")
        return json.dumps({"error": str(e)})


def call_with_retry(call_fn, max_retries=MAX_RETRIES):
    """
    Call a function with exponential backoff retry on transient errors.
    """
    for attempt in range(max_retries):
        try:
            return call_fn()
        except Exception as e:
            error_str = str(e).lower()
            is_transient = any(kw in error_str for kw in ["rate_limit", "429", "timeout", "503", "502"])
            if is_transient and attempt < max_retries - 1:
                delay = RETRY_BASE_DELAY * (2 ** attempt)
                logger.warning(f"Transient error (attempt {attempt + 1}/{max_retries}), retrying in {delay}s: {e}")
                time.sleep(delay)
            else:
                raise


def run_editorial_review(pptx_path: str) -> str:
    """
    Run the full editorial QA pipeline via the Foundry agent.
    
    Sends the initial request, then loops handling tool calls until
    the agent produces its final text response.
    
    Returns the agent's final output text.
    """
    # Reset accumulator for fresh run
    findings_accumulator.clear()
    extracted_deck_cache.clear()
    
    user_message = (
        f"Review the PowerPoint deck at path: {pptx_path}\n\n"
        "Follow your instructions exactly — you MUST call each tool:\n"
        "1. Call extract_deck with the PPTX path\n"
        "2. Call analyze_visual_consistency with the same PPTX path\n"
        "3. Call get_review_windows with the markdown\n"
        "4. For EACH chunk: review it, then call store_chunk_findings (MANDATORY — do NOT skip this tool)\n"
        "5. Take the 'findings_json' string from the analyze_visual_consistency result and call store_chunk_findings(chunk_id=-1, findings_json=<that string>) in ONE call (MANDATORY)\n"
        "6. Call merge_and_dedupe_findings\n"
        "7. Present the final editorial QA report\n\n"
        "CRITICAL: You must call store_chunk_findings for EVERY chunk. Do NOT summarize findings in text without storing them first.\n"
        "CRITICAL: Pass the findings_json from analyze_visual_consistency directly to store_chunk_findings — do NOT iterate or modify.\n"
    )
    
    logger.info(f"Starting editorial review for: {pptx_path}")
    
    # ── Initial call ──
    response = call_with_retry(lambda: openai_client.responses.create(
        input=user_message,
        extra_body={"agent_reference": {"name": agent.name, "version": str(agent.version), "type": "agent_reference"}},
    ))
    
    # ── Tool call loop ──
    for round_num in range(MAX_TOOL_ROUNDS):
        # Check if there are function calls in the output
        function_calls = [item for item in response.output if item.type == "function_call"]
        
        if not function_calls:
            # No more tool calls — agent is done
            logger.info(f"Agent completed after {round_num} tool rounds")
            break
        
        logger.info(f"Round {round_num + 1}: processing {len(function_calls)} tool call(s)")
        
        # Execute each function call and build the results list
        tool_results = []
        for fc in function_calls:
            logger.info(f"  → Calling tool: {fc.name}")
            result = execute_tool_call(fc.name, fc.arguments)
            
            # Log a preview of the result (truncated for readability)
            preview = result[:200] + "..." if len(result) > 200 else result
            logger.info(f"  ← Result preview: {preview}")
            
            tool_results.append(
                FunctionCallOutput(
                    type="function_call_output",
                    call_id=fc.call_id,
                    output=result,
                )
            )
        
        # Send tool results back to the agent
        response = call_with_retry(lambda: openai_client.responses.create(
            input=tool_results,
            previous_response_id=response.id,
            extra_body={"agent_reference": {"name": agent.name, "version": str(agent.version), "type": "agent_reference"}},
        ))
    else:
        logger.warning(f"Hit max tool rounds ({MAX_TOOL_ROUNDS}) — agent may not have finished")
    
    # Extract final text output
    final_output = response.output_text if hasattr(response, 'output_text') and response.output_text else ""
    
    if not final_output:
        # Try to get text from output items
        for item in response.output:
            if hasattr(item, 'text'):
                final_output = item.text
                break
            if hasattr(item, 'content'):
                for content in item.content:
                    if hasattr(content, 'text'):
                        final_output = content.text
                        break
    
    logger.info(f"Review complete. Final output length: {len(final_output)} chars")
    return final_output


print("✅ Orchestration functions defined")

✅ Orchestration functions defined


## Cell 10 — Run the Review & Display Report

Execute the full pipeline on a PPTX deck and display the formatted editorial QA report.

In [13]:
# ── Run the editorial review ──
PPTX_PATH = "test_decks/editorial_test.pptx"  # ← Original editorial test deck

assert os.path.exists(PPTX_PATH), f"Deck not found: {PPTX_PATH}"

# Reset accumulator for fresh run
findings_accumulator.clear()

print("🔍 Starting editorial QA review...")
print("═" * 60)

start_time = time.time()
final_report = run_editorial_review(PPTX_PATH)
elapsed = time.time() - start_time

print(f"\n⏱️  Completed in {elapsed:.1f} seconds")
print(f"📊 Total findings accumulated: {len(findings_accumulator)}")
print("═" * 60)

# ── Display the agent's formatted report ──
display(Markdown(final_report))

2026-03-21 20:14:51,011 [INFO] Starting editorial review for: test_decks/editorial_test.pptx


🔍 Starting editorial QA review...
════════════════════════════════════════════════════════════


2026-03-21 20:14:52,501 [INFO] HTTP Request: POST https://kd-foundry-resource.services.ai.azure.com/api/projects/kd-proj/openai/v1/responses "HTTP/1.1 200 OK"
2026-03-21 20:14:52,523 [INFO] Round 1: processing 2 tool call(s)
2026-03-21 20:14:52,524 [INFO]   → Calling tool: extract_deck
2026-03-21 20:14:52,553 [INFO]   ← Result preview: {"slide_count": 25, "word_count": 725, "estimated_tokens": 942, "markdown": "# Slide 1\nDigital Transformation Stratgy\nPrepared for Contoso Ltd. | March 2026\n\n\n# Slide 2\nExecutive Summery\nContos...
2026-03-21 20:14:52,554 [INFO]   → Calling tool: extract_deck_visual
2026-03-21 20:14:52,676 [INFO]   ← Result preview: {"slide_count": 25, "slides": [{"slide_num": 1, "shape_count": 2, "shapes": [{"shape_type": "PLACEHOLDER (14)", "name": "Title 1", "left_in": 0.75, "top_in": 2.33, "width_in": 8.5, "height_in": 1.61, ...
2026-03-21 20:15:00,780 [INFO] HTTP Request: POST https://kd-foundry-resource.services.ai.azure.com/api/projects/kd-proj/openai/v1/res


⏱️  Completed in 25.2 seconds
📊 Total findings accumulated: 19
════════════════════════════════════════════════════════════


### Editorial QA Report

#### High Severity Issues
1. **Slide 1:** 
   - **Evidence:** Digital Transformation Stratgy
   - **Issue:** 'Stratgy' is a misspelling of 'Strategy'.
   - **Remediation:** Change to "Strategy"
   - **Category:** Spelling

2. **Slide 2:** 
   - **Evidence:** Executive Summery
   - **Issue:** 'Summery' is a misspelling of 'Summary'.
   - **Remediation:** Change to "Summary"
   - **Category:** Spelling

3. **Slide 2:** 
   - **Evidence:** phased aproach
   - **Issue:** 'Aproach' is a misspelling of 'Approach'.
   - **Remediation:** Change to "Approach"
   - **Category:** Spelling

4. **Slide 3:** 
   - **Evidence:** Strategic Recomendations
   - **Issue:** 'Recomendations' is a misspelling of 'Recommendations'.
   - **Remediation:** Change to "Recommendations"
   - **Category:** Spelling

5. **Slide 4:** 
   - **Evidence:** The company have been
   - **Issue:** Incorrect verb form 'have'. Should be 'has'.
   - **Remediation:** Change to "The company has been"
   - **Category:** Grammar

6. **Slide 7:** 
   - **Evidence:** Manual processes creates bottlenecks
   - **Issue:** Subject-verb agreement error: 'processes creates'. Should be 'processes create'.
   - **Remediation:** Change to "Manual processes create bottlenecks"
   - **Category:** Grammar

7. **Slide 9:** 
   - **Evidence:** clear ownershp
   - **Issue:** 'Ownershp' is a misspelling of 'Ownership'.
   - **Remediation:** Change to "Ownership"
   - **Category:** Spelling

8. **Slide 11:** 
   - **Evidence:** Cloud Center of Excelence
   - **Issue:** Misspelling: 'Excelence' should be 'Excellence'.
   - **Remediation:** Change to "Excellence"
   - **Category:** Spelling

9. **Slide 13:** 
   - **Evidence:** digital intiatives
   - **Issue:** 'Intiatives' is a misspelling of 'Initiatives'.
   - **Remediation:** Change to "Initiatives"
   - **Category:** Spelling

10. **Slide 18:** 
    - **Evidence:** Contoso will provides
    - **Issue:** Incorrect verb form, should be 'provide'.
    - **Remediation:** Change to "Contoso will provide"
    - **Category:** Grammar

11. **Slide 19:** 
    - **Evidence:** effected teams
    - **Issue:** Incorrect word 'effected', should be 'affected'.
    - **Remediation:** Change to "affected teams"
    - **Category:** Grammar

12. **Slide 22:** 
    - **Evidence:** Their learnings has
    - **Issue:** Subject-verb agreement error: 'learnings has'. Should be 'learnings have'.
    - **Remediation:** Change to "Their learnings have"
    - **Category:** Grammar

13. **Slide 23:** 
    - **Evidence:** successfull cloud migrations
    - **Issue:** Misspelling: 'successfull' should be 'successful'.
    - **Remediation:** Change to "successful"
    - **Category:** Spelling

#### Medium Severity Issues

1. **Slides 1, 2:** 
   - **Evidence:** Title left positioning inconsistent.
   - **Issue:** Title left positioning is inconsistent across slides.
   - **Remediation:** Align title shapes to the same left position on all slides.
   - **Category:** Visual

2. **Slides 1, 2:** 
   - **Evidence:** Title width inconsistent.
   - **Issue:** Title width is inconsistent across slides.
   - **Remediation:** Make title shape width consistent across all slides.
   - **Category:** Visual

3. **Slide 5:** 
   - **Evidence:** Real time analytics
   - **Issue:** Inconsistent terminology. Should be 'real-time'.
   - **Remediation:** Change to "Real-time analytics"
   - **Category:** Terminology

4. **Slide 9:** 
   - **Evidence:** Casual paragraph tone
   - **Issue:** Informal tone in section where formal tone is expected.
   - **Remediation:** Revise tone for consistency with business formal context.
   - **Category:** Tone

5. **Slides 11, 13:** 
   - **Evidence:** Phase titles misaligned.
   - **Issue:** Phase titles are misaligned vertically across consecutive slides.
   - **Remediation:** Align phase titles consistently on all slides.
   - **Category:** Visual

6. **Slide 15:** 
   - **Evidence:** totally on board
   - **Issue:** Inconsistent tone; 'totally' is casual compared to surrounding formal business language.
   - **Remediation:** Change to "on board"
   - **Category:** Tone

This report includes the findings from both editorial and visual reviews, following strict guidelines and ensuring precision in identifying issues across the PowerPoint deck.

## Cell 10b — Visual Metadata Verification

Verifies that `extract_slide_visual_metadata` correctly captures planted inconsistencies in `test_decks/visual_test.pptx`.

Tests cover: font family/size deviations, color mismatches, bold/italic toggles, position/size shifts, fill colors, within-slide font mixing, within-slide color mixing, overlapping shapes, clutter (25 shapes), crammed margins, and rotated elements.

In [ ]:
# ── Verify visual metadata extraction matches planted issues in visual_test.pptx ──
meta = extract_slide_visual_metadata("test_decks/visual_test.pptx")

def get_title_font(slide_meta):
    """Get font of first (title) shape's first paragraph."""
    for sh in slide_meta['shapes']:
        if 'paragraphs' in sh:
            for p in sh['paragraphs']:
                if 'fonts' in p:
                    return p['fonts'][0]
    return {}

def get_body_font(slide_meta):
    """Get font of second text shape's first paragraph."""
    text_shapes = [sh for sh in slide_meta['shapes'] if 'paragraphs' in sh]
    if len(text_shapes) >= 2:
        for p in text_shapes[1]['paragraphs']:
            if 'fonts' in p:
                return p['fonts'][0]
    return {}

def get_title_shape(slide_meta):
    """Get first text shape (title)."""
    for sh in slide_meta['shapes']:
        if 'paragraphs' in sh:
            return sh
    return {}

passed = failed = 0
def check(name, cond, detail=""):
    global passed, failed
    s = "✅" if cond else "❌"
    if cond: passed += 1
    else: failed += 1
    print(f"  {s} {name}" + (f" — {detail}" if detail and not cond else ""))

print(f"Total slides: {len(meta)}\n")

# Slide 3 - Title Arial
f = get_title_font(meta[2])
check("Slide 3: Title font = Arial", f.get('name') == 'Arial', f"got {f.get('name')}")
check("Slide 3: Title size = 32pt", f.get('size_pt') == 32.0, f"got {f.get('size_pt')}")

# Slide 4 - Body Times New Roman
f = get_body_font(meta[3])
check("Slide 4: Body font = Times New Roman", f.get('name') == 'Times New Roman', f"got {f.get('name')}")

# Slide 5 - Title Georgia
f = get_title_font(meta[4])
check("Slide 5: Title font = Georgia", f.get('name') == 'Georgia', f"got {f.get('name')}")

# Slide 6 - Title 28pt
f = get_title_font(meta[5])
check("Slide 6: Title size = 28pt", f.get('size_pt') == 28.0, f"got {f.get('size_pt')}")

# Slide 7 - Body 14pt
f = get_body_font(meta[6])
check("Slide 7: Body size = 14pt", f.get('size_pt') == 14.0, f"got {f.get('size_pt')}")

# Slide 8 - Title 24pt
f = get_title_font(meta[7])
check("Slide 8: Title size = 24pt", f.get('size_pt') == 24.0, f"got {f.get('size_pt')}")

# Slide 9 - Title color blue
f = get_title_font(meta[8])
check("Slide 9: Title color = 2563EB", f.get('color') == '2563EB', f"got {f.get('color')}")

# Slide 10 - Body color black
f = get_body_font(meta[9])
check("Slide 10: Body color = 000000", f.get('color') == '000000', f"got {f.get('color')}")

# Slide 11 - Title color red
f = get_title_font(meta[10])
check("Slide 11: Title color = DC2626", f.get('color') == 'DC2626', f"got {f.get('color')}")

# Slide 12 - Title NOT bold
f = get_title_font(meta[11])
check("Slide 12: Title bold = False", f.get('bold') == False, f"got {f.get('bold')}")

# Slide 13 - Body italic
f = get_body_font(meta[12])
check("Slide 13: Body italic = True", f.get('italic') == True, f"got {f.get('italic')}")

# Slide 15 - Title top = 1.2
sh = get_title_shape(meta[14])
check("Slide 15: Title top = 1.2\"", sh.get('top_in') == 1.2, f"got {sh.get('top_in')}")

# Slide 16 - Title left = 2.5
sh = get_title_shape(meta[15])
check("Slide 16: Title left = 2.5\"", sh.get('left_in') == 2.5, f"got {sh.get('left_in')}")

# Slide 17 - Title width = 7.0
sh = get_title_shape(meta[16])
check("Slide 17: Title width = 7.0\"", sh.get('width_in') == 7.0, f"got {sh.get('width_in')}")

# Slide 18 - Accent rect green fill
shapes_18 = meta[17]['shapes']
fill_colors = [sh.get('fill_color') for sh in shapes_18 if sh.get('fill_color')]
check("Slide 18: Has green fill", any('16A34A' in c for c in fill_colors), f"fills={fill_colors}")

# Slide 19 - Within-slide font mixing
text_shapes_19 = [sh for sh in meta[18]['shapes'] if 'paragraphs' in sh]
if len(text_shapes_19) >= 3:
    f1 = text_shapes_19[1].get('paragraphs', [{}])[0].get('fonts', [{}])[0]
    f2 = text_shapes_19[2].get('paragraphs', [{}])[0].get('fonts', [{}])[0]
    check("Slide 19: Left box = Arial 20pt", f1.get('name') == 'Arial' and f1.get('size_pt') == 20.0)
    check("Slide 19: Right box = Calibri 16pt", f2.get('name') == 'Calibri' and f2.get('size_pt') == 16.0)

# Slide 21 - Within-slide color mixing
text_shapes_21 = [sh for sh in meta[20]['shapes'] if 'paragraphs' in sh]
if len(text_shapes_21) >= 3:
    f1 = text_shapes_21[1].get('paragraphs', [{}])[0].get('fonts', [{}])[0]
    f2 = text_shapes_21[2].get('paragraphs', [{}])[0].get('fonts', [{}])[0]
    check("Slide 21: Left box color = 000000", f1.get('color') == '000000')
    check("Slide 21: Right box color = 64748B", f2.get('color') == '64748B')

# Slide 22 - Overlapping
shapes_22 = [sh for sh in meta[21]['shapes'] if 'paragraphs' in sh]
check("Slide 22: Has 4+ text shapes (overlapping)", len(shapes_22) >= 4, f"got {len(shapes_22)}")

# Slide 23 - Clutter
check("Slide 23: 25 shapes", meta[22]['shape_count'] == 25, f"got {meta[22]['shape_count']}")

# Slide 24 - Crammed + Misaligned + Rotation
sh_24 = meta[23]['shapes']
has_rotation = any(sh.get('rotation', 0) != 0 for sh in sh_24)
check("Slide 24: Has rotated shape", has_rotation)
has_crammed = any(sh.get('left_in', 1) < 0.3 for sh in sh_24)
check("Slide 24: Has crammed shape (left<0.3\")", has_crammed)

print(f"\n{'='*40}")
print(f"Results: {passed}/{passed+failed} passed, {failed}/{passed+failed} failed")
if failed == 0:
    print("🎉 All metadata extractions verified!")

Total slides: 25

  ✅ Slide 3: Title font = Arial
  ✅ Slide 3: Title size = 32pt
  ✅ Slide 4: Body font = Times New Roman
  ✅ Slide 5: Title font = Georgia
  ✅ Slide 6: Title size = 28pt
  ✅ Slide 7: Body size = 14pt
  ✅ Slide 8: Title size = 24pt
  ✅ Slide 9: Title color = 2563EB
  ✅ Slide 10: Body color = 000000
  ✅ Slide 11: Title color = DC2626
  ✅ Slide 12: Title bold = False
  ✅ Slide 13: Body italic = True
  ✅ Slide 15: Title top = 1.2"
  ✅ Slide 16: Title left = 2.5"
  ✅ Slide 17: Title width = 7.0"
  ✅ Slide 18: Has green fill
  ✅ Slide 19: Left box = Arial 20pt
  ✅ Slide 19: Right box = Calibri 16pt
  ✅ Slide 21: Left box color = 000000
  ✅ Slide 21: Right box color = 64748B
  ✅ Slide 22: Has 4+ text shapes (overlapping)
  ✅ Slide 23: 25 shapes
  ✅ Slide 24: Has rotated shape
  ✅ Slide 24: Has crammed shape (left<0.3")

Results: 24/24 passed, 0/24 failed
🎉 All metadata extractions verified!


: 

## Cell 11 — Post-Run Analysis & Findings Summary

Parse and display the merged findings with statistics.

In [14]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 11: Post-Run Analysis & Findings Summary
# ═══════════════════════════════════════════════════════════════════════════════

from tabulate import tabulate

# ── Get the merged/deduped findings ──
merged_json = merge_and_dedupe_findings()
merged = json.loads(merged_json)

print("═" * 60)
print("EDITORIAL QA FINDINGS — SUMMARY")
print("═" * 60)
print(f"Total raw findings: {len(findings_accumulator)}")
print(f"After deduplication: {merged['total']}")
print(f"Duplicates removed: {merged['duplicates_removed']}")

# ── Severity breakdown ──
severity_counts = {"Critical": 0, "High": 0, "Medium": 0, "Low": 0}
for f in merged["merged_findings"]:
    sev = f.get("flag", f.get("severity", "Low"))
    if sev in severity_counts:
        severity_counts[sev] += 1

print("\n📊 Severity Breakdown:")
for sev, count in severity_counts.items():
    bar = "█" * count
    print(f"  {sev:>10}: {count:>3}  {bar}")

# ── Category breakdown ──
category_counts: Dict[str, int] = {}
for f in merged["merged_findings"]:
    cat = f.get("category", "Unknown")
    category_counts[cat] = category_counts.get(cat, 0) + 1

print("\n📋 Category Breakdown:")
for cat, count in sorted(category_counts.items(), key=lambda x: -x[1]):
    print(f"  {cat:>20}: {count}")

# ── Findings table ──
if merged["merged_findings"]:
    print("\n" + "═" * 60)
    print("DETAILED FINDINGS")
    print("═" * 60)
    
    table_rows = []
    for f in merged["merged_findings"]:
        table_rows.append([
            f.get("slides", f.get("slide", "—")),
            f.get("issue", "—")[:60],
            f.get("flag", f.get("severity", "—")),
            f.get("remediation", "—")[:60],
        ])
    
    print(tabulate(
        table_rows,
        headers=["Slide(s)", "Issue", "Flag", "Remediation"],
        tablefmt="grid",
        maxcolwidths=[12, 60, 10, 60],
    ))
else:
    print("\n✨ No issues found — deck looks clean!")

════════════════════════════════════════════════════════════
EDITORIAL QA FINDINGS — SUMMARY
════════════════════════════════════════════════════════════
Total raw findings: 19
After deduplication: 19
Duplicates removed: 0

📊 Severity Breakdown:
    Critical:   0  
        High:  13  █████████████
      Medium:   6  ██████
         Low:   0  

📋 Category Breakdown:
              Spelling: 8
               Grammar: 5
                Visual: 3
                  Tone: 2
           Terminology: 1

════════════════════════════════════════════════════════════
DETAILED FINDINGS
════════════════════════════════════════════════════════════
+-------------+--------------------------------------------------------------+--------+-------------------------------------------------------------+
| Slide(s)    | Issue                                                        | Flag   | Remediation                                                 |
+=============+==============================================

## Cell 12 — Validation Tests

Automated checks to verify the pipeline meets acceptance criteria.

In [15]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 12: Validation Tests
# ═══════════════════════════════════════════════════════════════════════════════

def run_validation_tests():
    """Run validation checks on the most recent pipeline execution."""
    print("═" * 60)
    print("VALIDATION TESTS")
    print("═" * 60)
    
    passed = 0
    failed = 0
    
    def check(name: str, condition: bool, detail: str = ""):
        nonlocal passed, failed
        status = "✅ PASS" if condition else "❌ FAIL"
        print(f"  {status}: {name}")
        if detail and not condition:
            print(f"         → {detail}")
        if condition:
            passed += 1
        else:
            failed += 1
    
    # ── Test 1: Extraction produced slides ──
    if PPTX_PATH in extracted_deck_cache:
        deck_md = extracted_deck_cache[PPTX_PATH]
        slide_markers = re.findall(r"# Slide \d+", deck_md)
        check("Extraction produced slide markers", len(slide_markers) > 0,
              f"Found {len(slide_markers)} markers")
        check("Slide 1 marker present", "# Slide 1" in deck_md)
    else:
        check("Extraction cache populated", False, "No deck in cache")
    
    # ── Test 2: No speaker notes in output ──
    if PPTX_PATH in extracted_deck_cache:
        deck_md = extracted_deck_cache[PPTX_PATH]
        has_notes_marker = "speaker note" in deck_md.lower() or "notes:" in deck_md.lower()
        check("No speaker notes in extraction", not has_notes_marker,
              "Found 'speaker note' or 'notes:' in output")
    
    # ── Test 3: Chunking validation ──
    if PPTX_PATH in extracted_deck_cache:
        deck_md = extracted_deck_cache[PPTX_PATH]
        chunks = adaptive_chunk(deck_md)
        check("Chunking produced at least 1 chunk", len(chunks) >= 1,
              f"Got {len(chunks)} chunks")
        all_have_markers = all(re.search(r"# Slide \d+", c[2]) for c in chunks)
        check("All chunks contain slide markers", all_have_markers)
    
    # ── Test 4: Findings have required fields ──
    merged_json = merge_and_dedupe_findings()
    merged = json.loads(merged_json)
    if merged["merged_findings"]:
        sample = merged["merged_findings"][0]
        has_slides = "slides" in sample or "slide" in sample
        has_issue = "issue" in sample
        has_flag = "flag" in sample or "severity" in sample
        has_remediation = "remediation" in sample
        check("Findings have slide citations", has_slides)
        check("Findings have issue descriptions", has_issue)
        check("Findings have severity flags", has_flag)
        check("Findings have remediation", has_remediation)
        
        # Check all findings have slide citations
        all_cited = all(
            ("slides" in f or "slide" in f) for f in merged["merged_findings"]
        )
        check("ALL findings have slide citations", all_cited)
        
        # Check valid severity values
        valid_severities = {"Critical", "High", "Medium", "Low"}
        all_valid_sev = all(
            f.get("flag", f.get("severity", "")) in valid_severities
            for f in merged["merged_findings"]
        )
        check("All severity flags are valid", all_valid_sev)
    else:
        check("Findings produced", False, "No findings in merged output")
    
    # ── Test 5: Deduplication works ──
    check("Deduplication ran", merged["total"] <= len(findings_accumulator),
          f"Merged: {merged['total']}, Raw: {len(findings_accumulator)}")
    
    # ── Summary ──
    print("\n" + "─" * 40)
    total = passed + failed
    print(f"Results: {passed}/{total} passed, {failed}/{total} failed")
    if failed == 0:
        print("🎉 All validation tests passed!")
    else:
        print("⚠️  Some tests failed — review the output above")


run_validation_tests()

2026-03-21 20:15:25,485 [INFO] Single-pass mode: 942 estimated tokens (threshold: 90,000)


════════════════════════════════════════════════════════════
VALIDATION TESTS
════════════════════════════════════════════════════════════
  ✅ PASS: Extraction produced slide markers
  ✅ PASS: Slide 1 marker present
  ✅ PASS: No speaker notes in extraction
  ✅ PASS: Chunking produced at least 1 chunk
  ✅ PASS: All chunks contain slide markers
  ✅ PASS: Findings have slide citations
  ✅ PASS: Findings have issue descriptions
  ✅ PASS: Findings have severity flags
  ✅ PASS: Findings have remediation
  ✅ PASS: ALL findings have slide citations
  ✅ PASS: All severity flags are valid
  ✅ PASS: Deduplication ran

────────────────────────────────────────
Results: 12/12 passed, 0/12 failed
🎉 All validation tests passed!


## Cell 13 — Cleanup

Delete the agent when you're done to avoid leaving orphaned resources.

In [21]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 13: Cleanup
# ═══════════════════════════════════════════════════════════════════════════════

# Uncomment the lines below to clean up resources when you're done.

# project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
# print(f"🧹 Agent '{agent.name}' (version {agent.version}) deleted")

# openai_client.close()
# project_client.close()
# credential.close()
# print("🧹 Clients closed")